In [1]:
%load_ext autoreload
%autoreload 2

import dataclasses
import os
os.environ["JAX_PLATFORMS"] = "cpu,cuda"
from functools import partial

import moe
from tests import utils as test_utils

import jax
import jax.numpy as jnp
from jax.sharding import PartitionSpec as P
import tune_jax
import numpy as np

try:
  jax.config.update("jax_num_cpu_devices", 4)
except RuntimeError:
  print("CPU devices already set")

tune_jax.logger.setLevel("INFO")

# profile the moe block

In [2]:
axis_name = "x"
devices = jax.devices("cuda")
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)
jax.set_mesh(mesh)
keys = iter(jax.random.split(jax.random.key(0), 10000))
g = 32
x = jax.random.normal(next(keys), (4096, 2048), dtype="bfloat16")
idx = jax.random.randint(next(keys), x.shape[0], minval=0, maxval=g)
x = jax.device_put(x, P("x", None))
idx = jax.device_put(idx)

In [22]:
run_moe = jax.jit(partial(moe.core.run_moe, axis_name=axis_name, experts_num=g, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all))
run_moe_multiple = jax.jit(partial(moe.core.run_moe, axis_name=axis_name, experts_num=g, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all, multiple=8))

y1 = run_moe(x, idx)
y2 = run_moe_multiple(x, idx)
print(jnp.sum(jnp.abs((y1 - y2))))

vjp1 = jax.jit(jax.vjp(partial(run_moe, all_idxs=idx), x)[1])
vjp2 = jax.jit(jax.vjp(partial(run_moe_multiple, all_idxs=idx), x)[1])

r = jax.device_put(jax.random.normal(next(keys), y1.shape, dtype=y1.dtype), y1.sharding)
dy1 = vjp1(r)
dy2 = vjp2(r)

jax.block_until_ready(run_moe(x, idx))
jax.block_until_ready(run_moe_multiple(x, idx))
with moe.utils.profile():
  for _ in range(3):
    jax.block_until_ready(run_moe(x, idx))
  for _ in range(3):
    jax.block_until_ready(run_moe_multiple(x, idx))
  for _ in range(3):
    jax.block_until_ready(vjp1(r))
  for _ in range(3):
    jax.block_until_ready(vjp2(r))

0
http://localhost:52550/data/plugin/profile/trace_viewer@;run=2025_11_22_15_41_22;tag=trace_viewer@


In [20]:
jnp.sum(jnp.abs(dy2[0] - dy1[0]))

Array(0, dtype=bfloat16)

# testing ra2a simulator

In [10]:
axis_name = "x"
devices = jax.devices()
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=jax.sharding.AxisType.Explicit, devices=devices)
jax.set_mesh(mesh)

x, meta = test_utils.generate_data(4096, 1024, len(devices), axis_name=axis_name)

out = jax.device_put(jnp.zeros_like(x, shape=(x.shape[0] * 2,) + x.shape[1:]), P("x", None))
# out = jnp.zeros_like(x, shape=(x.shape[0] * 2,) + x.shape[1:])


@partial(jax.shard_map, out_specs=x.sharding.spec)
def fn(x, out, meta):
  return moe.ra2a_simulator.ragged_all_to_all(x, out, *dataclasses.astuple(meta), axis_name=axis_name)


out = fn(x, out, meta)

# custom gather testing

### inplace def

In [19]:
@partial(jax.tree_util.register_dataclass, meta_fields=[], data_fields=[
  "group_counts", "group_counts_with_padding", "group_idx", "group_idx_with_padding",
  "sort_idx", "isort_idx", "inv_sort_idx"
])
@dataclasses.dataclass
class PaddedGroupPaddedMetadata:
  group_idx: jax.Array
  group_idx_with_padding: jax.Array
  group_counts: jax.Array
  group_counts_with_padding: jax.Array
  sort_idx: jax.Array
  isort_idx: jax.Array
  inv_sort_idx: jax.Array


def compute_padded_group_gather(group_idx: jax.Array, groups: int, multiple: int) -> PaddedGroupPaddedMetadata:
  assert multiple >= 1
  group_counts = jnp.bincount(group_idx, length=groups)

  if multiple != 1:
    padding_idxs = moe.utils.add_indices(jnp.arange(groups), -group_counts % multiple, max_size=multiple - 1)
    group_idx_with_padding = jnp.concat([group_idx, padding_idxs], axis=0)
    group_counts_with_padding = group_counts + (-group_counts % multiple)
  else:
    group_idx_with_padding, group_counts_with_padding = group_idx, group_counts
  sort_idx = jnp.argsort(group_idx_with_padding)
  isort_idx = jnp.argsort(sort_idx)[:group_idx.shape[0]]
  inv_sort_idx = isort_idx[:group_idx.shape[0]]

  return PaddedGroupPaddedMetadata(
    group_idx, group_idx_with_padding, group_counts, group_counts_with_padding,
    sort_idx, isort_idx, inv_sort_idx
  )


@jax.custom_vjp
def custom_gather(x: jax.Array, idx: jax.Array, inv_idx: jax.Array):
  return x[idx, ...]


def custom_gather_fwd(x: jax.Array, idx: jax.Array, inv_idx: jax.Array):
  return custom_gather(x, idx, inv_idx), (x.shape, inv_idx,)


def custom_gather_bwd(res, g):
  (x_shape, inv_idx,) = res
  if x_shape[0] <= inv_idx.size:  # gather
    grad = g[inv_idx, ...]
  else:  # scatter
    grad = jnp.zeros_like(g, shape=x_shape).at[inv_idx, ...].set(g, mode="drop")
  return (grad, None, None)


custom_gather.defvjp(custom_gather_fwd, custom_gather_bwd)

### testing

In [2]:
def test_fn(x, group_idx):
  sort = jnp.argsort(group_idx)
  isort = jnp.argsort(sort)
  y = x[sort, ...]
  z = y[isort, ...]
  # y = custom_gather(x, sort, isort)
  # z = custom_gather(y, isort, sort)
  return z


def test_fn2(x, group_idx):
  info = moe.utils.compute_padded_group_gather(group_idx, groups=8, multiple=1)
  print(jax.tree.map(jax.typeof, info))
  # y = x[info.sort_idx, ...]
  # z = y[info.isort_idx[:x.shape[0]], ...]
  y = moe.utils.custom_gather(x, info.sort_idx, info.inv_sort_idx, mode="gather")
  # z = y[info.isort_idx[:x.shape[0]], ...]
  # z = custom_gather(y, info.isort_idx[:x.shape[0]], info.inv_sort_idx)
  z = moe.utils.custom_gather(y, info.isort_idx, info.inv_sort_idx, mode="scatter")
  # z = custom_scatter(y, info.isort_idx[:x.shape[0]], info.inv_sort_idx)
  return z

In [3]:
keys = iter(jax.random.split(jax.random.key(0), 1024))
x = jnp.arange(16)[:, None] * jnp.ones((1, 4))
group_idx = jax.random.randint(next(keys), x.shape[0], minval=0, maxval=8)
print(group_idx)
r = jax.random.normal(next(keys), x.shape, dtype=x.dtype)

[7 1 7 1 2 0 1 5 5 4 5 1 7 3 5 5]


In [7]:
vjp1 = jax.vjp(partial(test_fn, group_idx=group_idx), x)[1]
vjp2 = jax.vjp(partial(test_fn2, group_idx=group_idx), x)[1]

PaddedGroupPaddedMetadata(group_idx=ShapedArray(int32[16]), group_idx_with_padding=ShapedArray(int32[16]), group_counts=ShapedArray(int32[8]), group_counts_with_padding=ShapedArray(int32[8]), sort_idx=ShapedArray(int32[16]), isort_idx=ShapedArray(int32[16]), inv_sort_idx=ShapedArray(int32[16]))


In [8]:
# jax.make_jaxpr(jax.grad(lambda x: jnp.sum(test_fn(x, group_idx=group_idx))))(x)

In [9]:
vjp1(r)[0]

Array([[-2.4424558 , -2.0356805 ,  0.20554423, -0.3535502 ],
       [-0.76197404, -1.1785518 , -1.1482196 ,  0.29716578],
       [-1.3105359 ,  2.1302025 , -0.18957235,  0.96401215],
       [-1.3011001 , -0.7486938 , -0.3729984 ,  0.4427907 ],
       [-1.1902995 , -0.06925564, -0.95605886, -1.9587638 ],
       [-1.1059942 , -0.3220052 ,  1.3303299 ,  0.81223685],
       [-1.1588238 , -0.5432413 ,  0.8078304 ,  1.6546043 ],
       [-0.40930986,  0.05388845, -0.64838815, -1.7675956 ],
       [-0.21153021,  0.9224308 , -1.2988783 , -0.7148775 ],
       [-2.4024377 , -1.6770942 , -0.4698358 , -1.7149593 ],
       [-1.9887297 , -1.6507224 , -0.27444926, -0.71897036],
       [ 0.39305303, -0.9410713 ,  1.9166322 ,  0.55670065],
       [-0.07911822,  0.7670797 ,  0.9159606 , -1.1321445 ],
       [ 0.69038856, -1.3735921 ,  0.1859908 , -1.6042006 ],
       [-0.4742109 ,  0.5001767 , -0.7488765 ,  0.9681463 ],
       [-0.30875412,  0.01906552,  1.056175  ,  0.9335578 ]],      dtype=float32)

In [10]:
vjp2(r)[0]

Array([[-2.4424558 , -2.0356805 ,  0.20554423, -0.3535502 ],
       [-0.76197404, -1.1785518 , -1.1482196 ,  0.29716578],
       [-1.3105359 ,  2.1302025 , -0.18957235,  0.96401215],
       [-1.3011001 , -0.7486938 , -0.3729984 ,  0.4427907 ],
       [-1.1902995 , -0.06925564, -0.95605886, -1.9587638 ],
       [-1.1059942 , -0.3220052 ,  1.3303299 ,  0.81223685],
       [-1.1588238 , -0.5432413 ,  0.8078304 ,  1.6546043 ],
       [-0.40930986,  0.05388845, -0.64838815, -1.7675956 ],
       [-0.21153021,  0.9224308 , -1.2988783 , -0.7148775 ],
       [-2.4024377 , -1.6770942 , -0.4698358 , -1.7149593 ],
       [-1.9887297 , -1.6507224 , -0.27444926, -0.71897036],
       [ 0.39305303, -0.9410713 ,  1.9166322 ,  0.55670065],
       [-0.07911822,  0.7670797 ,  0.9159606 , -1.1321445 ],
       [ 0.69038856, -1.3735921 ,  0.1859908 , -1.6042006 ],
       [-0.4742109 ,  0.5001767 , -0.7488765 ,  0.9681463 ],
       [-0.30875412,  0.01906552,  1.056175  ,  0.9335578 ]],      dtype=float32)

In [11]:
vjp1(r)[0] - vjp2(r)[0]

Array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]], dtype=float32)

# group padded gather

In [12]:
x = jax.random.normal(jax.random.key(0), (24, 1024))
idx = jax.random.randint(jax.random.key(0), (24,), minval=0, maxval=8)


def fn(x):
  return moe.utils.padded_group_gather(x, idx, groups=8, multiple=4).y


# dout = jnp.ones_like(x) * jnp.arange(x.shape[0])[:, None] + 7
dout = jax.random.normal(jax.random.key(0), x.shape, dtype=x.dtype)

In [24]:
jax.grad(lambda *args: jnp.sum(fn(*args)))(x) == jnp.ones_like(x)

Array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True]], dtype=bool)

In [13]:
o, vjp_fn = jax.vjp(lambda *args: fn(*args), x)
dout_ = dout[moe.utils.scatter_arange(o.shape[0], jnp.bincount(idx), 4)[0]][:, 0]

In [14]:
dout_ = moe.utils.padded_group_gather(dout, idx, groups=8, multiple=4).y

In [18]:
vjp_fn(dout_)[0][:, 0]

Array([-0.1681759 , -1.8616787 ,  0.27153736,  0.3265926 ,  0.81122196,
       -0.57248914,  1.7422539 , -0.31242236, -1.2161462 ,  0.38323358,
        0.8282944 , -0.9738775 ,  1.4084821 , -1.6718844 , -1.351549  ,
       -0.5939988 ,  0.2744671 ,  0.6407501 ,  1.6226422 ,  1.8875649 ,
       -1.4279947 ,  0.60068065, -1.2371411 , -0.95844775], dtype=float32)

In [19]:
jax.jvp(lambda x: x[jnp.argsort(idx), ...], (x,), (dout,))[1][:, 0]

Array([-0.1681759 , -1.8616787 ,  0.27153736,  0.3265926 ,  0.81122196,
       -0.57248914,  1.7422539 , -0.31242236, -1.2161462 ,  0.38323358,
        0.8282944 , -0.9738775 ,  1.4084821 , -1.6718844 , -1.351549  ,
       -0.5939988 ,  0.2744671 ,  0.6407501 ,  1.6226422 ,  1.8875649 ,
       -1.4279947 ,  0.60068065, -1.2371411 , -0.95844775], dtype=float32)

In [ ]:
dout[:, 0]

In [ ]:
counts = jnp.bincount(idx, length=8)
padding_idxs = moe.utils.add_indices(jnp.arange(8), -counts % 4, max_size=4 - 1)
idx_with_padding = jnp.concat([idx, padding_idxs], axis=0)
gather_idx = jnp.argsort(idx_with_padding)

In [ ]:
dout_[:, 0][jnp.argsort(gather_idx)[jnp.argsort(idx)]]

In [ ]:
r = jax.random.normal(jax.random.key(0), (x.shape[0],))


def fn1(x, idx):
  y = x[jnp.argsort(idx), ...]
  print(r)
  print(y)
  return jnp.sum(y * r[:, None])


def fn2(x, idx):
  y = moe.utils.padded_group_gather(x, idx, max_idx=8, multiple=4)
  r_idx, mask = moe.utils.scatter_arange(y.shape[0], jnp.bincount(idx), multiple=4)
  r_ = r[r_idx, ...] * mask
  print(mask)
  print(r_)
  print(y)
  return jnp.sum(y * r_[:, None])



In [ ]:
y = fn(x)
y

In [ ]:
fn1(x, idx)

In [ ]:
fn2(x, idx)

In [ ]:
jax.jvp(fn, (x,), (dout,))[1][0][:, 0]

In [ ]:
m, k, n = (8192, 4096, 128)
s = jnp.ones((m, k), "bfloat16")
v = jnp.ones((k, n), "bfloat16")

fn = jax.jit(jnp.dot)

with moe.utils.profile():
  for _ in range(3):
    jax.block_until_ready(fn(s, v))
  for _ in range(3):
    jax.block_until_ready(fn(v.T, s.T))

In [ ]:
jnp.bincount(moe.utils.add_indices(jnp.array([1, 3, 5]), jnp.array([0, 7, 2]), 10), length=16)

In [ ]:
jnp.bincount(jnp.arange(12), length=2)

# actual moe testing

In [12]:
devices = jax.devices("cpu")
axis_name = "x"
mesh = jax.make_mesh((len(devices),), (axis_name,), axis_types=(jax.sharding.AxisType.Explicit,), devices=devices)
jax.sharding.set_mesh(mesh)

In [21]:
# x, meta = test_utils.generate_data(4096, 7168, device_num=len(devices), axis_name="x")
x = jax.device_put(jax.random.normal(jax.random.key(1), (256, 1024), dtype="bfloat16"), P(axis_name, None))

In [22]:
g = 32
all_idxs = jax.device_put(jax.random.randint(jax.random.key(0), (1 * x.shape[0],), minval=0, maxval=g), P(None))
# run_moe = partial(moe.core.run_moe, axis_name="x", experts_num=g, multiple=8, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all)
# run_moe2 = partial(moe.core.run_moe, axis_name="x", experts_num=g, multiple=8, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all, custom_gathers=True)
run_moe = partial(moe.core.run_moe, axis_name="x", experts_num=g, multiple=2, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all)
run_moe2 = partial(moe.core.run_moe, axis_name="x", experts_num=g, multiple=2, ragged_all_to_all=moe.ra2a_simulator.ragged_all_to_all, custom_gathers=True)

x2 = run_moe(x, all_idxs)
x3 = run_moe2(x, all_idxs)

print(jnp.sum(jnp.abs(jnp.sum(x2[:, :1, ...], axis=1) - x)))
print(jnp.sum(jnp.abs(jnp.sum(x3[:, :1, ...], axis=1) - x)))

0
0


In [23]:
vjp_fn = jax.vjp(partial(run_moe, all_idxs=all_idxs), x)[1]
vjp_fn2 = jax.vjp(partial(run_moe2, all_idxs=all_idxs), x)[1]
r = jax.device_put(jax.random.normal(jax.random.key(0), x2.shape, dtype=x2.dtype), x2.sharding)

In [24]:
out1 = vjp_fn(r)[0]
out2 = vjp_fn2(r)[0]

In [25]:
out1

Array([[0.386719, 0.182617, -1, ..., -0.212891, -2.89062, 0.515625],
       [-0.271484, -1, -0.375, ..., -0.251953, -1.30469, 0.182617],
       [-0.9375, -0.152344, 0.261719, ..., -1.35156, -1.83594, -2.32812],
       ...,
       [-1.83594, 1.4375, -1.95312, ..., 0.632812, -1.73438, 0.539062],
       [-0.417969, -2.32812, -0.0539551, ..., -1, 0.142578, -0.824219],
       [-0.972656, 0.365234, 1.24219, ..., 0.365234, 0.386719, -0.597656]],      dtype=bfloat16)

In [26]:
out2

Array([[0.386719, 0.182617, -1, ..., -0.212891, -2.89062, 0.515625],
       [-0.271484, -1, -0.375, ..., -0.251953, -1.30469, 0.182617],
       [-0.9375, -0.152344, 0.261719, ..., -1.35156, -1.83594, -2.32812],
       ...,
       [-1.83594, 1.4375, -1.95312, ..., 0.632812, -1.73438, 0.539062],
       [-0.417969, -2.32812, -0.0539551, ..., -1, 0.142578, -0.824219],
       [-0.972656, 0.365234, 1.24219, ..., 0.365234, 0.386719, -0.597656]],      dtype=bfloat16)

In [27]:
jnp.sum(jnp.abs(jnp.sum(r, axis=1) - out1), axis=-1) / jnp.sum(jnp.abs(out1), axis=-1)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=bfloat16)

In [28]:
jnp.sum(jnp.abs(out1 - out2), axis=-1)

Array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=bfloat16)

# mesh experiments

In [ ]:
n = jax.device_count()
mesh = jax.make_mesh((n,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.sharding.set_mesh(mesh)

In [ ]:
fn = lambda: moe.utils.empty((1024, 1024), jnp.bfloat16, P(None, "x"))

In [ ]:
fn_ = tune_jax.tune(fn)
fn_()

# ra2a 2d

In [ ]:
n_devices = jax.device_count()
mesh = jax.make_mesh((n_devices,), ("x",), axis_types=(jax.sharding.AxisType.Explicit,))
jax.set_mesh(mesh)

In [ ]:
x_sort, input_offsets, send_sizes, output_offsets, recv_sizes = test_utils.generate_data(8 * 8 * 4096, 4096, n_devices, multiple=8)
x_sort = x_sort.reshape((x_sort.shape[0], -1)).astype(jnp.bfloat16)



In [ ]:
@jax.jit
@partial(jax.shard_map, out_specs=P("x", None), check_vma=False)
def test_kernel_async(x, input_offsets, send_sizes, output_offsets, recv_sizes):
  # output = jnp.zeros((2 * x.shape[0],) + x.shape[1:], x.dtype)
  output = moe.utils.empty((2 * x.shape[0],) + x.shape[1:], x.dtype)
  with jax.named_scope("start"):
    out = moe.ra2a.ra2a_2d(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x", multiple=8)
  with jax.named_scope("jax.lax.ragged_all_to_all"):
    out2 = jax.lax.ragged_all_to_all(x, output, input_offsets, send_sizes, output_offsets, recv_sizes, axis_name="x")
  return out, out2



In [ ]:
out, out2 = test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes)
err = jnp.sum(jnp.abs(out - out2))
print(f"{err = }")

In [ ]:
with jax.profiler.trace("/tmp/ra2a"):
  for _ in range(3):
    out, out2 = jax.block_until_ready(test_kernel_async(x_sort, input_offsets, send_sizes, output_offsets, recv_sizes))

# compute on

In [2]:
import jax
import jax.numpy as jnp
from jax.experimental.compute_on import compute_on


@jax.jit
@compute_on("tpu_sparsecore")
def my_gather(x, idx):
  return x[idx, ...]


x = jnp.ones((8192, 4096))
idx = jnp.argsort(np.random.randn(x.shape[0]))

In [3]:
with jax.profiler.trace("/tmp/compute_on"):
  for _ in range(3):
    jax.block_until_ready(my_gather(x, idx))